# Day 4 Lab: Delta Lake Pipeline in Databricks

## Objectives
- Understand Delta Lake
- Perform MERGE (Upserts)
- Use Time Travel
- Build reliable pipelines


In [ ]:
orders_df = spark.read.option('header', True).csv('dbfs:/FileStore/labs/orders.csv')
display(orders_df)

In [ ]:
orders_df.write.format('delta').mode('overwrite').save('/tmp/delta/orders')

In [ ]:
delta_df = spark.read.format('delta').load('/tmp/delta/orders')
display(delta_df)

In [ ]:
new_data = [
 (7,101,'2024-01-06',300,'completed'),
 (2,102,'2024-01-02',350,'completed')
]
new_df = spark.createDataFrame(new_data, ['order_id','customer_id','order_date','amount','status'])
display(new_df)

In [ ]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, '/tmp/delta/orders')

delta_table.alias('target').merge(
    new_df.alias('source'),
    'target.order_id = source.order_id'
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

In [ ]:
display(delta_table.toDF())

In [ ]:
spark.read.format('delta').option('versionAsOf', 0).load('/tmp/delta/orders')

In [ ]:
bad_data = [(8, 'invalid_customer', 'bad_date', 'wrong_amount', 'completed')]
bad_df = spark.createDataFrame(bad_data)
bad_df.write.format('delta').mode('append').save('/tmp/delta/orders')

In [ ]:
spark.sql("OPTIMIZE delta.`/tmp/delta/orders`")